# Part 2 — Bayesian Probability

**Dataset:** IMDb Movie Reviews (50,000 reviews, balanced 25,000 positive / 25,000 negative)
**Chosen conditional:** we compute **P(Positive | keyword)** *(not P(Negative | keyword))*

**Keywords**

| Type | Keywords |
|---|---|
| Positive | `masterpiece`, `hilarious`, `brilliant` |
| Negative | `boring`, `disappointing`, `waste of time` |
| Control  | `funny` |

The **control keyword** `funny` is deliberately sentiment-ambiguous ("funny ha-ha" is positive,
"the plot felt funny/off" is negative). We expect its posterior to land near the 0.5 baseline,
which lets us sanity-check that the method is actually picking up sentiment signal rather than noise.

> Constraint: **basic Python only** — standard-library `csv` and `string`, no pandas and no
> machine-learning libraries. Every probability is computed by hand from raw counts.


## The mathematics: Bayes' Theorem

We want the probability that a review is **positive** *given that it contains a keyword*:

$$P(\text{Positive} \mid \text{keyword}) = \frac{P(\text{keyword} \mid \text{Positive}) \; P(\text{Positive})}{P(\text{keyword})}$$

We treat each **review** as one observation and ask only whether the keyword appears in it (at
least once) or not. The four quantities, expressed as raw review counts:

| Term | Meaning | Count formula |
|---|---|---|
| **Prior** $P(\text{Positive})$ | how common positive reviews are | positive reviews / total reviews |
| **Likelihood** $P(\text{keyword} \mid \text{Positive})$ | how often the keyword shows up *inside* positive reviews | (positive **and** keyword) / positive reviews |
| **Marginal** $P(\text{keyword})$ | how common the keyword is overall | reviews with keyword / total reviews |
| **Posterior** $P(\text{Positive} \mid \text{keyword})$ | what we want | Likelihood × Prior / Marginal |

Because the dataset is balanced, the prior is exactly **0.5** — so the posterior reads cleanly
against a 0.5 baseline: **above 0.5 → the keyword leans positive, below 0.5 → leans negative.**


## 1. Configuration

In [1]:
# Basic Python only: csv + string from the standard library. No pandas, no ML libraries.

DATASET_PATH = "IMDB_Dataset.csv"   # keep the CSV in the same folder as this notebook

POSITIVE_KEYWORDS = ["masterpiece", "hilarious", "brilliant"]
NEGATIVE_KEYWORDS = ["boring", "disappointing", "waste of time"]
CONTROL_KEYWORDS  = ["funny"]

ALL_KEYWORDS = POSITIVE_KEYWORDS + NEGATIVE_KEYWORDS + CONTROL_KEYWORDS
ALL_KEYWORDS

['masterpiece',
 'hilarious',
 'brilliant',
 'boring',
 'disappointing',
 'waste of time',
 'funny']

## 2. Load the data

In [2]:
import csv


def load_reviews(path):
    """Load the IMDb dataset into a list of (review_text, sentiment) tuples."""
    reviews = []
    with open(path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            reviews.append((row["review"], row["sentiment"]))
    return reviews


reviews = load_reviews(DATASET_PATH)
print(f"Loaded {len(reviews):,} reviews")
print("Example sentiment label:", reviews[0][1])

Loaded 50,000 reviews
Example sentiment label: positive


## 3. Text-matching helpers

Whole-word matching for single keywords (so `brilliant` does **not** fire on `brilliantly`),
and substring matching for the multi-word phrase `waste of time`. All matching is
case-insensitive and ignores the `<br />` HTML tags that appear in the raw reviews.

In [3]:
import string


def tokenize(text):
    """Lowercase, drop HTML breaks, turn punctuation into spaces, and return the
    SET of unique words in the review. Used for whole-word keyword matching."""
    text = text.lower().replace("<br />", " ")
    for ch in string.punctuation:
        text = text.replace(ch, " ")
    return set(text.split())


def clean_text(text):
    """Lowercased, single-spaced version of the review. Used for phrase matching
    (needed for the multi-word keyword 'waste of time')."""
    text = text.lower().replace("<br />", " ")
    return " ".join(text.split())


def review_contains(keyword, tokens, text):
    """Whole-word match for single words; substring match for multi-word phrases."""
    if " " in keyword:            # multi-word phrase -> substring on cleaned text
        return keyword in text
    return keyword in tokens      # single word -> exact whole-word (token) match

## 4. Count occurrences (single pass, DRY)

One loop over the dataset collects everything the Bayes formula needs: the total number of
reviews, the number of positive reviews, and — per keyword — how many positive reviews contain
it and how many reviews contain it overall.

In [4]:
def count_occurrences(reviews, keywords):
    """Single pass over the data. Returns:
        total       - number of reviews
        n_positive  - number of positive reviews
        kw_and_pos  - {keyword: reviews that are positive AND contain the keyword}
        kw_total    - {keyword: reviews that contain the keyword, any sentiment}
    """
    total = 0
    n_positive = 0
    kw_and_pos = {k: 0 for k in keywords}
    kw_total = {k: 0 for k in keywords}

    for review, sentiment in reviews:
        total += 1
        is_positive = (sentiment == "positive")
        if is_positive:
            n_positive += 1

        tokens = tokenize(review)
        text = clean_text(review)
        for k in keywords:
            if review_contains(k, tokens, text):
                kw_total[k] += 1
                if is_positive:
                    kw_and_pos[k] += 1

    return total, n_positive, kw_and_pos, kw_total


total, n_positive, kw_and_pos, kw_total = count_occurrences(reviews, ALL_KEYWORDS)
print(f"Total reviews:    {total:,}")
print(f"Positive reviews: {n_positive:,}")
print(f"Prior P(Positive) = {n_positive / total:.4f}")

Total reviews:    50,000
Positive reviews: 25,000
Prior P(Positive) = 0.5000


## 5. Bayes' Theorem (one modular function)

A single function computes all four quantities for any keyword, so there is no repetition
across the seven keywords (DRY).

In [5]:
def bayes_positive_given_keyword(keyword, total, n_positive, kw_and_pos, kw_total):
    """Compute the four Bayesian quantities for P(Positive | keyword).

        Prior       P(Positive)         = n_positive / total
        Likelihood  P(keyword|Positive) = (positive AND keyword) / n_positive
        Marginal    P(keyword)          = (reviews with keyword) / total
        Posterior   P(Positive|keyword) = Likelihood * Prior / Marginal
    """
    prior = n_positive / total
    likelihood = kw_and_pos[keyword] / n_positive
    marginal = kw_total[keyword] / total
    posterior = (likelihood * prior) / marginal if marginal > 0 else 0.0

    return {
        "keyword": keyword,
        "prior": prior,
        "likelihood": likelihood,
        "marginal": marginal,
        "posterior": posterior,
    }


results = [
    bayes_positive_given_keyword(k, total, n_positive, kw_and_pos, kw_total)
    for k in ALL_KEYWORDS
]
print("Computed Bayes results for:", [r["keyword"] for r in results])

Computed Bayes results for: ['masterpiece', 'hilarious', 'brilliant', 'boring', 'disappointing', 'waste of time', 'funny']


## 6. Per-keyword probability blocks

The assignment asks for a small table for **each** keyword showing the prior, likelihood,
marginal, and posterior.

In [6]:
from IPython.display import Markdown, display


def format_keyword_block(r):
    """Return a small markdown table of the four probabilities for one keyword."""
    return (
        f"**Keyword: `{r['keyword']}`**\n\n"
        f"| Quantity | Formula | Value |\n"
        f"|---|---|---|\n"
        f"| Prior      | P(Positive)             | {r['prior']:.4f} |\n"
        f"| Likelihood | P(keyword \\| Positive)   | {r['likelihood']:.5f} |\n"
        f"| Marginal   | P(keyword)              | {r['marginal']:.5f} |\n"
        f"| **Posterior** | **P(Positive \\| keyword)** | **{r['posterior']:.4f}** |\n"
    )


for r in results:
    display(Markdown(format_keyword_block(r)))

**Keyword: `masterpiece`**

| Quantity | Formula | Value |
|---|---|---|
| Prior      | P(Positive)             | 0.5000 |
| Likelihood | P(keyword \| Positive)   | 0.03512 |
| Marginal   | P(keyword)              | 0.02414 |
| **Posterior** | **P(Positive \| keyword)** | **0.7274** |


**Keyword: `hilarious`**

| Quantity | Formula | Value |
|---|---|---|
| Prior      | P(Positive)             | 0.5000 |
| Likelihood | P(keyword \| Positive)   | 0.04956 |
| Marginal   | P(keyword)              | 0.03788 |
| **Posterior** | **P(Positive \| keyword)** | **0.6542** |


**Keyword: `brilliant`**

| Quantity | Formula | Value |
|---|---|---|
| Prior      | P(Positive)             | 0.5000 |
| Likelihood | P(keyword \| Positive)   | 0.06348 |
| Marginal   | P(keyword)              | 0.04176 |
| **Posterior** | **P(Positive \| keyword)** | **0.7601** |


**Keyword: `boring`**

| Quantity | Formula | Value |
|---|---|---|
| Prior      | P(Positive)             | 0.5000 |
| Likelihood | P(keyword \| Positive)   | 0.02368 |
| Marginal   | P(keyword)              | 0.06104 |
| **Posterior** | **P(Positive \| keyword)** | **0.1940** |


**Keyword: `disappointing`**

| Quantity | Formula | Value |
|---|---|---|
| Prior      | P(Positive)             | 0.5000 |
| Likelihood | P(keyword \| Positive)   | 0.00684 |
| Marginal   | P(keyword)              | 0.01572 |
| **Posterior** | **P(Positive \| keyword)** | **0.2176** |


**Keyword: `waste of time`**

| Quantity | Formula | Value |
|---|---|---|
| Prior      | P(Positive)             | 0.5000 |
| Likelihood | P(keyword \| Positive)   | 0.00124 |
| Marginal   | P(keyword)              | 0.01384 |
| **Posterior** | **P(Positive \| keyword)** | **0.0448** |


**Keyword: `funny`**

| Quantity | Formula | Value |
|---|---|---|
| Prior      | P(Positive)             | 0.5000 |
| Likelihood | P(keyword \| Positive)   | 0.12356 |
| Marginal   | P(keyword)              | 0.12750 |
| **Posterior** | **P(Positive \| keyword)** | **0.4845** |


## 7. Summary table (all keywords)

In [7]:
def print_summary(results):
    """Aligned text summary of every keyword (also works outside a notebook)."""
    header = (f"{'keyword':<15}{'Prior':>8}{'Likelihood':>12}"
              f"{'Marginal':>10}{'Posterior':>11}   leans")
    print(header)
    print("-" * len(header))
    for r in results:
        leans = "POSITIVE" if r["posterior"] > 0.5 else "negative"
        print(f"{r['keyword']:<15}{r['prior']:>8.3f}{r['likelihood']:>12.5f}"
              f"{r['marginal']:>10.5f}{r['posterior']:>11.4f}   {leans}")


print_summary(results)

keyword           Prior  Likelihood  Marginal  Posterior   leans
----------------------------------------------------------------
masterpiece       0.500     0.03512   0.02414     0.7274   POSITIVE
hilarious         0.500     0.04956   0.03788     0.6542   POSITIVE
brilliant         0.500     0.06348   0.04176     0.7601   POSITIVE
boring            0.500     0.02368   0.06104     0.1940   negative
disappointing     0.500     0.00684   0.01572     0.2176   negative
waste of time     0.500     0.00124   0.01384     0.0448   negative
funny             0.500     0.12356   0.12750     0.4845   negative


## 8. Sanity check

The posterior can also be read straight off the counts as
*(positive reviews with the keyword) / (all reviews with the keyword)*.
If the Bayes implementation is correct, the two must match exactly.

In [8]:
for r in results:
    k = r["keyword"]
    direct = kw_and_pos[k] / kw_total[k]
    assert abs(direct - r["posterior"]) < 1e-9, f"mismatch for {k}"
    print(f"{k:<15} Bayes={r['posterior']:.5f}   direct={direct:.5f}   OK")

print("\nAll posteriors match the direct count — Bayes' Theorem implemented correctly.")

masterpiece     Bayes=0.72742   direct=0.72742   OK
hilarious       Bayes=0.65417   direct=0.65417   OK
brilliant       Bayes=0.76006   direct=0.76006   OK
boring          Bayes=0.19397   direct=0.19397   OK
disappointing   Bayes=0.21756   direct=0.21756   OK
waste of time   Bayes=0.04480   direct=0.04480   OK
funny           Bayes=0.48455   direct=0.48455   OK

All posteriors match the direct count — Bayes' Theorem implemented correctly.


## 9. Interpretation

*(Values below are produced by the code above; a balanced dataset makes the prior exactly 0.5,
so every posterior can be compared directly against that 0.5 baseline.)*

**Positive keywords — all pull the posterior above 0.5.**
`brilliant` (~0.76) and `masterpiece` (~0.73) are the strongest positive signals: when a review
contains them it is roughly three times as likely to be positive as negative. `hilarious` (~0.66)
(~0.65) is positive but a little weaker, because comedy can also show up in negative reviews
complaining that a film was *unintentionally* hilarious.

**Negative keywords — all pull the posterior below 0.5.**
`waste of time` (~0.045) is the single strongest signal in the whole set: it appears almost
exclusively in negative reviews. `disappointing` (~0.22) and `boring` (~0.19) are clearly
negative as well. Notice `boring` is a *common* word (high marginal) yet still strongly negative
— frequency and sentiment strength are independent.

**Control keyword `funny` — lands almost exactly on the baseline (~0.48).**
This is the key result. `funny` is the most frequent keyword we chose, but its posterior sits
essentially at the 0.5 prior, meaning it carries almost **no** sentiment signal on its own —
exactly what we predicted for an ambiguous word. This confirms the method is genuinely detecting
sentiment (the loaded words move the posterior; the neutral word does not) rather than reacting
to how often a word appears.

**Why P(Positive | keyword) and not both?** The two are complementary —
P(Negative | keyword) = 1 − P(Positive | keyword) — so computing one gives the other for free,
and the assignment asks us to commit to a single direction.
